[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced/blob/main/CH05/05_02_begin/notebooks/05_02_train_the_assistant.ipynb)

# 05_02 Train the support assistant

Fine-tune Qwen3 4B Instruct on the Ridgeline training set, watch the loss, test the result, and save your adapter.

Before you start, choose **Runtime > Change runtime type > T4 GPU**.

In [ ]:
%%capture
# Install Unsloth. This takes a minute or two.
!pip install unsloth

In [ ]:
# Pull the course files into this Colab session and load the Ridgeline helpers.
!git clone --quiet https://github.com/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced.git course 2>/dev/null || echo "Course files already downloaded."

import sys
sys.path.insert(0, "course/CH05")

import ridgeline

## Load the model and add LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                   # rank: how much room the adapter has to learn
    lora_alpha=16,          # scaling: how strongly the adapter's update counts
    lora_dropout=0,         # Unsloth is optimized for 0
    target_modules=[        # the layers that get adapters
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",      # feed-forward
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",       # saves memory
    random_state=3407,
)
model.print_trainable_parameters()

## Load the training data

Each conversation gets rendered with the model's chat template, the same way you saw in Chapter 4.

In [ ]:
# TODO: Load train.jsonl and eval.jsonl with load_dataset and render each conversation into a text field with ridgeline.chat_text.


## Set up the trainer

These are the settings from the last video.

In [ ]:
from trl import SFTConfig
from unsloth import is_bfloat16_supported

config = SFTConfig(
    output_dir="outputs",
    dataset_text_field="text",
    max_length=2048,
    per_device_train_batch_size=2,     # examples per step on the GPU
    gradient_accumulation_steps=4,     # 2 x 4 = 8 examples per update
    num_train_epochs=2,               # passes through the training data
    learning_rate=2e-4,              # how big each update step is
    warmup_steps=5,                    # ease into the learning rate
    lr_scheduler_type="linear",
    optim="adamw_8bit",                # memory-efficient optimizer
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    per_device_eval_batch_size=4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    seed=3407,
    report_to="none",
)

In [ ]:
# TODO: Create the SFTTrainer and wrap it with train_on_responses_only, using the Qwen user and assistant markers.


## Measure the starting point

In [ ]:
# TODO: Run trainer.evaluate() and print the evaluation loss.


## Train

In [ ]:
# TODO: Run trainer.train() and print how long it took.


## Read the loss curves

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_points = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_points = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.plot(*zip(*train_points), label="training loss")
plt.plot(*zip(*eval_points), marker="o", label="evaluation loss")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

## Test it on tickets it has never seen

In [ ]:
FastLanguageModel.for_inference(model)
tickets = {t["id"]: t for t in ridgeline.load_tickets()}

results = []
for ticket_id in ["RT-001", "RT-008", "RT-017"]:
    ticket = tickets[ticket_id]
    # No voice rules in the prompt. The model has to have learned them.
    messages = ridgeline.build_messages(ticket, include_voice_rules=False)
    reply = ridgeline.generate(model, tokenizer, messages)
    print(f"--- {ticket_id}: {ticket['subject']} ---")
    print(reply, "\n")
    result = ridgeline.check_reply(ticket, reply)
    ridgeline.print_check(result)
    results.append(result)
    print()

## Save your adapter to Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

ADAPTER_DIR = "/content/drive/MyDrive/ridgeline-lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved your LoRA adapter to {ADAPTER_DIR}")